### Saliency Detection

- 显著性检测（Saliency Detection），根本目的是模拟人类的视觉注意力机制（Human Visual System, HVS），从图像或视频中自动定位并提取出最吸引人眼球、最具视觉刺激的区域或目标。
- 信息论视角的“惊喜度” (Information Maximization)
    - 从信息论的视角，显著性等于**“惊喜度” (Surprise)** 或**“自信息量” (Self-Information)**。
如果一幅图像中 90% 的像素都是蓝天，那么“蓝色”出现的概率 $p(x)$ 极高，提供的信息熵极低。而一只飞鸟的像素概率极低，其蕴含的信息量极大。在数学上，位置 $(x,y)$ 处的视觉特征 $F$ 的显著性 $S$ 可被严谨地定义为特征概率的负对数：
        - $S(x,y) = -\log\big(p(F(x,y))\big)$
- 输入输出以及应用
    - 输入：$(B, C, H, W)$，其中 $C=3$ 代表红、绿、蓝三个颜色通道。
    - 输出：是一个单通道的显著性图 (Saliency Map)。其维度通常与输入图像的空间分辨率一致，为 $(B, 1, H, W)$。

### Spectral Residual (侯晓迪 2007 cvpr)

https://github.com/uoip/SpectralResidualSaliency/blob/master/src/saliency.py

- 很多研究者执迷于在空域 (Spatial Domain) 卷复杂的像素对比度和形态学闭运算。但在纯粹的数学变换视角下，自然图像转换到频域后，其对数振幅谱 $L(f)$ 具有极其相似的统计冗余规律。如果用原始振幅谱减去平滑后的平均振幅谱 $h_n(f) * L(f)$，得到的残差谱 (Spectral Residual, SR) 就包含了图像中的“创新信息”（即显著目标所在）：

$$R(f) = L(f) - h_n(f) * L(f)$$

- 仅仅通过极少量的代码（图像作快速傅里叶变换 $\to$ 计算残差 $\to$ 傅里叶反变换回空域），就能优雅且极速地获得高质量的先验显著图。

```python
c = cv2.dft(np.float32(img), flags = cv2.DFT_COMPLEX_OUTPUT)
mag = np.sqrt(c[:,:,0]**2 + c[:,:,1]**2)
```
- 通过离散傅里叶变换（DFT），将二维图像从“空间域”映射到了“频率域”。输出的 c 包含了复数的实部（c[:,:,0]）和虚部（c[:,:,1]）。随后计算出幅度谱（Magnitude Spectrum）$|F(u,v)|$。
    - cv2.dft
        - 该语句的输出是一个 Shape 为 (H, W, 2) 的 NumPy 数组，数据类型保持为浮点型（float32）。
        - 通道 0 (output[:, :, 0])：存储复数的实部（Real Part）
        - 通道 1 (output[:, :, 1])：存储复数的虚部（Imaginary Part）。
```python
spectralResidual = np.exp(np.log(mag) - cv2.boxFilter(np.log(mag), -1, (3,3)))
```
- 自然图像的幅度谱在对数尺度下，具有一种近乎普遍的规律（$1/f$ 定律），即大部分能量集中在低频，且呈现出平滑下降的趋势。
- 对数幅度谱 $L(f)$： np.log(mag)。
- 均值滤波 $A(f)$： cv2.boxFilter(np.log(mag), -1, (3,3)) 代表用一个 $3\times3$ 的滤波器去平滑对数幅度谱，它模拟了这幅图像的**“背景冗余信息”**（即人类视觉系统“预期”会看到的平庸模式）。
- 计算残差 $R(f)$： $R(f) = L(f) - A(f)$。用实际的频谱减去平滑后的预期频谱，剩下的就是“出乎意料”的高优信息。

```python
c[:,:,0] = c[:,:,0] * spectralResidual / mag
c[:,:,1] = c[:,:,1] * spectralResidual / mag
```
- 傅里叶变换不仅有幅度谱（大小），还有相位谱（方向/位置）。c[:,:,0] / mag 和 c[:,:,1] / mag 实质上是在提取原始图像的相位信息 $\cos(\theta)$ 和 $\sin(\theta)$。然后，将上一步计算出的“残差幅度”与原始的“相位”相乘，构造出一个全新的、剔除了背景冗余的复数频域表示。